In [1]:
import os
import re
import glob
import base64
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
from configparser import ConfigParser
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# OAuth Scopes
SCOPES = ['https://www.googleapis.com/auth/gmail.compose']

def load_config():
    """Load configuration from INI file"""
    config = ConfigParser()
    config.optionxform = str  # Preserve case sensitivity
    config.read('config.ini')
    return config

def authenticate_gmail():
    """More robust authentication with error handling"""
    creds = None
    token_file = 'token.json'
    
    # 1. Check for existing token
    if os.path.exists(token_file):
        try:
            creds = Credentials.from_authorized_user_file(token_file, SCOPES)
            
            # Check if token needs refresh
            if creds and creds.expired and creds.refresh_token:
                print("Refreshing access token...")
                creds.refresh(Request())
                # Save refreshed token
                with open(token_file, 'w') as token:
                    token.write(creds.to_json())
                return build('gmail', 'v1', credentials=creds)
                
            elif creds and creds.valid:
                return build('gmail', 'v1', credentials=creds)
                
        except Exception as e:
            print(f"Token load/refresh failed: {e}")
            os.remove(token_file)
    
    # 2. Fresh authentication flow
    print("Starting new authentication flow...")
    try:
        flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
        creds = flow.run_local_server(port=0, timeout_seconds=60)
        
        # Save the new token
        with open(token_file, 'w') as token:
            token.write(creds.to_json())
            
        return build('gmail', 'v1', credentials=creds)
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        raise

def get_latest_report_folder(base_path):
    """Find the latest report folder (max year/max date) and return its datetime"""
    if not os.path.exists(base_path):
        raise FileNotFoundError(f"Reports base path not found: {base_path}")
    
    # Find latest year
    years = [d for d in os.listdir(base_path) if d.isdigit() and os.path.isdir(os.path.join(base_path, d))]
    if not years:
        raise FileNotFoundError("No year folders found in reports base path")
    
    latest_year = max(years)
    year_path = os.path.join(base_path, latest_year)
    
    # Find latest date in year
    dates = [d for d in os.listdir(year_path) if d.isdigit() and os.path.isdir(os.path.join(year_path, d))]
    if not dates:
        raise FileNotFoundError(f"No date folders found in {year_path}")
    
    latest_date = max(dates)
    folder_path = os.path.join(year_path, latest_date)
    folder_date = datetime.strptime(latest_date, "%Y%m%d")
    
    return folder_path, folder_date

def get_client_reports(folder_path, client_code, report_types):
    """Find all matching report PDFs for a client"""
    reports = []
    for report_type in report_types:
        pattern = f"*{client_code}*{report_type}.pdf"
        matches = glob.glob(os.path.join(folder_path, pattern))
        reports.extend(matches)
    return reports

def create_performance_email(config, client_code, client_name, client_emails, report_files, folder_date):
    """Create a draft performance report email"""
    try:
        # Prepare email components using folder date
        month_year = folder_date.strftime("%b-%Y")
        
        # Determine performance email text based on report types
        report_types = [os.path.basename(f).split('_')[-1].replace('.pdf', '') for f in report_files]
        if 'TOW' in report_types and 'MOSL' in report_types:
            subject = config['EMAIL']['performance_report_subject'].format(
                month=folder_date.strftime("%b"),
                year=folder_date.strftime("%Y"),
                suubject='Performance Report and Holding Statement'
            )
            performance_text = "Please find your portfolio performance report by our system and the holding statement by MOSL attached with this email."
        elif 'MOSL' in report_types:
            subject = config['EMAIL']['performance_report_subject'].format(
                month=folder_date.strftime("%b"),
                year=folder_date.strftime("%Y"),
                suubject='Holding Statement'
            )
            performance_text = "Please find the holding statement by MOSL attached with this email."
        else:
            subject = config['EMAIL']['performance_report_subject'].format(
                month=folder_date.strftime("%b"),
                year=folder_date.strftime("%Y"),
                suubject='Holding Statement'
            )
            performance_text = "Please find the attached performance reports."
        
        # Format body with proper newlines
        body_template = config['EMAIL']['performance_report_body'].replace('\\n', '\n')
        body = body_template.format(
            client_names=client_name,
            performance_email_text=performance_text
        )
        
        # Create email message
        msg = MIMEMultipart()
        msg['Subject'] = subject
        msg['From'] = config['EMAIL']['sender']
        msg['To'] = client_emails
        
        # Attach body text
        msg.attach(MIMEText(body))
        
        # Attach report files
        for report_file in report_files:
            with open(report_file, 'rb') as f:
                part = MIMEApplication(f.read(), Name=os.path.basename(report_file))
            part['Content-Disposition'] = f'attachment; filename="{os.path.basename(report_file)}"'
            msg.attach(part)
        
        return msg
    
    except Exception as e:
        print(f"Error creating email for {client_code}: {str(e)}")
        return None

def main():
    try:
        # Load configuration
        config = load_config()
        
        # Get reports base path and latest folder date
        reports_base = config['SETTINGS']['reports_base']
        latest_folder, folder_date = get_latest_report_folder(reports_base)
        print(f"Using reports from: {latest_folder} (Date: {folder_date.strftime('%d-%b-%Y')})")
        
        # Get client mappings
        client_emails = dict(config['CLIENTS'])
        client_names = dict(config['CLIENT_NAMES']) if config.has_section('CLIENT_NAMES') else {}
        report_clients = dict(config['MONTHEND_REPORTS']) if config.has_section('MONTHEND_REPORTS') else {}
        
        # Authenticate with Gmail
        try:
            service = authenticate_gmail()
        except Exception as e:
            print(f"Fatal authentication error: {e}")
            # Optionally: send email notification to admin
            sys.exit(1)
        
        # Process each client needing reports
        for client_code, report_types_str in report_clients.items():
            # Get client details
            emails = client_emails.get(client_code, '').strip()
            name = client_names.get(client_code, 'Sir/Madam')
            report_types = [rt.strip() for rt in report_types_str.split(',')]
            
            if not emails:
                print(f"Skipping client {client_code} - no email configured")
                continue
            
            # Find matching report files
            report_files = get_client_reports(latest_folder, client_code, report_types)
            if not report_files:
                print(f"No reports found for client {client_code}")
                continue
            
            # Create and save draft
            email_msg = create_performance_email(config, client_code, name, emails, report_files, folder_date)
            if email_msg:
                raw_message = base64.urlsafe_b64encode(email_msg.as_bytes()).decode()
                draft = {'message': {'raw': raw_message}}
                created_draft = service.users().drafts().create(
                    userId='me',
                    body=draft
                ).execute()
                print(f"Draft created for {client_code} ({name}) with {len(report_files)} reports")
                
    except Exception as e:
        print(f"An error occurred: {str(e)}")

if __name__ == '__main__':
    main()

Using reports from: C:\\MyDocuments\\03Business\\06ClientData\\PMS\\ClientReports\2026\20260227 (Date: 27-Feb-2026)
Refreshing access token...
Token load/refresh failed: ('invalid_grant: Bad Request', {'error': 'invalid_grant', 'error_description': 'Bad Request'})
Starting new authentication flow...
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=419767357911-f8981vl1dljg5hcajssq9ud96p1c3loe.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A55608%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.compose&state=bhc7mWSlVRPBcbpYN1yqS5L5S5oUww&access_type=offline
Draft created for H0822 (Shubham Pawar) with 2 reports
Draft created for H1837 (Jayshree/Hemant Chaudhari) with 2 reports
Draft created for H2031 (Anurag Pawar) with 2 reports
Draft created for H20404 (Rohini Kulaye) with 2 reports
Draft created for H20405 (Madhavi Kulaye) with 2 reports
Draft created for H20427 (Yatin Gawde) with 2 r